# 🟢 Simulador de Hackeo - Interface Matrix

Sistema interactivo inspirado en **The Matrix** que utiliza **Python**, **OpenCV** y **MediaPipe Hands** para controlar un juego mediante gestos de la mano.

El jugador debe capturar las letras objetivo realizando un gesto de pinza (pulgar + índice) mientras una lluvia de caracteres estilo Matrix cae por la pantalla.

---

## Características

- Lluvia de caracteres estilo Matrix.
- Detección de manos en tiempo real.
- Control mediante gestos.
- Ingreso del nombre del jugador.
- Tres niveles de dificultad.
- Cronómetro por nivel.
- Tiempo total de la partida.
- Ranking de mejores tiempos.
- Botón virtual para salir del juego.

---

## Gestos

| Gesto | Acción |
|--------|--------|
| 🤏 Pinza (Pulgar + Índice) | Selecciona la letra objetivo |
| ✋ Mano completamente abierta | Pausa el juego |
| 🤏 Pinza sobre botón **INICIAR** | Comienza la partida |
| 🤏 Pinza sobre botón **NUEVO JUEGO** | Reinicia el juego |
| 🤏 Pinza sobre botón **SALIR** | Cierra la aplicación |

---

## Niveles

El jugador debe completar las siguientes palabras:

1. UNIBE
2. UNIVERSIDAD
3. INGENIERIA

Cada nivel aumenta la velocidad de caída de las letras.

---

## Controles

- Escribir el nombre con el teclado.
- **Enter** → Iniciar juego.
- **Backspace** → Borrar caracteres.
- **ESC** → Salir de la aplicación.

---

## Tecnologías

- Python
- OpenCV
- MediaPipe Hands
- NumPy
- Tkinter

---

> **Objetivo:** Aplicar visión por computadora y reconocimiento de gestos para crear una experiencia interactiva inspirada en la interfaz de *The Matrix*, utilizando la cámara web como método de control.

In [ ]:
pip install opencv-python mediapipe numpy cv2

In [ ]:
import cv2
import mediapipe as mp
import random
import numpy as np
import math
import time
import tkinter as tk

In [ ]:
# --- CONFIGURACIÓN DE LA MATRIX Y JUEGO ---
CHAR_LIST = "1234567890ABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%^&*"
WIDTH, HEIGHT = 640, 480
NUM_COLUMNS = 60
VELOCITY_BASE = 5

# Colores (BGR)
GREEN = (0, 255, 0)
WHITE = (200, 255, 200)
RED_TARGET = (0, 0, 255)
YELLOW = (0, 255, 255)
DARK_BG = (0, 0, 0)

LEVEL_WORDS = ["UNIBE", "UNIVERSIDAD", "INGENIERIA"]

# --- FUNCIONES AUXILIARES ---
def draw_centered_text(img, text, font, scale, color, thickness, y_pos):
    text_size = cv2.getTextSize(text, font, scale, thickness)[0]
    text_x = (WIDTH - text_size[0]) // 2
    cv2.putText(img, text, (text_x, y_pos), font, scale, color, thickness)

def get_pinch_point(hand_landmarks, width, height):
    pulgar = hand_landmarks.landmark[4]
    indice = hand_landmarks.landmark[8]
    dist = math.hypot((indice.x * width) - (pulgar.x * width), 
                      (indice.y * height) - (pulgar.y * height))
    if dist < 40:
        px = int((pulgar.x + indice.x) / 2 * width)
        py = int((pulgar.y + indice.y) / 2 * height)
        return px, py
    return None

def is_hand_open(hand_landmarks, width, height):
    pulgar = hand_landmarks.landmark[4]
    indice = hand_landmarks.landmark[8]
    if math.hypot((indice.x * width) - (pulgar.x * width), (indice.y * height) - (pulgar.y * height)) < 60:
        return False
        
    tips = [8, 12, 16, 20]
    mcp = [5, 9, 13, 17]
    dedos_estirados = sum([1 for t, m in zip(tips, mcp) if hand_landmarks.landmark[t].y < hand_landmarks.landmark[m].y])
    return dedos_estirados == 4

def check_button_click(px, py, btn_x, btn_y, btn_w, btn_h):
    return btn_x < px < btn_x + btn_w and btn_y < py < btn_y + btn_h

# --- CLASES ---
class Drop:
    def __init__(self, x):
        self.x = x
        self.reset()
    def reset(self):
        self.y = random.randint(-HEIGHT, 0)
        self.velocity = random.uniform(VELOCITY_BASE, VELOCITY_BASE + 3)
        self.chars = [random.choice(CHAR_LIST) for _ in range(random.randint(5, 12))]
        self.length = len(self.chars)
    def update(self):
        self.y += self.velocity
        if self.y > HEIGHT + 100:
            self.reset()

class TargetLetter:
    def __init__(self, char, level):
        self.char = char
        self.level = level
        self.reset()
    def reset(self):
        self.x = random.choice([i * (WIDTH // NUM_COLUMNS) for i in range(NUM_COLUMNS)])
        self.y = random.randint(-150, -50)
        
        if self.level == 0: self.velocity = random.uniform(6, 8)
        elif self.level == 1: self.velocity = random.uniform(9, 12)
        else: self.velocity = random.uniform(14, 18)

    def update(self):
        self.y += self.velocity
        if self.y > HEIGHT + 50:
            self.reset()

class GameSystem:
    def __init__(self):
        self.columns = [Drop(i * (WIDTH // NUM_COLUMNS)) for i in range(NUM_COLUMNS)]
        
        # Sistema de Estados
        self.state = "MENU"
        self.player_name = ""
        self.history = [] # Tuplas (Nombre, Tiempo Total)
        
        # Variables de tiempo
        self.level_start_time = 0
        self.level_times = []
        self.current_level = 0
        self.transition_timer = 0
        
        # Botones
        self.btn_start = (WIDTH//2 - 75, HEIGHT//2 + 50, 150, 40)
        self.btn_restart = (WIDTH//2 - 100, HEIGHT - 80, 200, 40)

    def start_game(self):
        if len(self.player_name.strip()) > 0:
            self.current_level = 0
            self.level_times = []
            self.setup_level()

    def reset_to_menu(self):
        self.player_name = ""
        self.state = "MENU"

    def setup_level(self):
        # Si aún hay niveles disponibles, lo preparamos y pasamos al estado PLAYING
        if self.current_level < len(LEVEL_WORDS):
            self.current_word = LEVEL_WORDS[self.current_level]
            self.current_char_index = 0
            if self.current_word[self.current_char_index] == " ":
                self.current_char_index += 1
            self.target = TargetLetter(self.current_word[self.current_char_index], self.current_level)
            self.level_start_time = time.time()
            self.state = "PLAYING" # <-- CORRECCIÓN: Definimos el estado aquí
        else:
            # Si ya no hay niveles, guardamos el historial y pasamos al estado END
            total_time = sum(self.level_times)
            self.history.append((self.player_name, total_time))
            self.history.sort(key=lambda x: x[1])
            self.state = "END" # <-- CORRECCIÓN: Salto directo al final

    def draw_rain(self, matrix_frame, is_paused=False):
        for col in self.columns:
            if not is_paused:
                col.update()
            for i, char in enumerate(col.chars):
                color = GREEN
                if i == 0: color = WHITE
                elif i < col.length // 2:
                    alpha = int(255 * (col.length - i) / col.length)
                    color = (0, alpha, 0)
                y_pos = int(col.y + i * 20)
                if 0 < y_pos < HEIGHT:
                    cv2.putText(matrix_frame, char, (int(col.x), y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

    def update_and_draw_game(self, matrix_frame, frame, pinch_points, is_paused):
        # --- ESTADO: MENÚ INICIAL ---
        if self.state == "MENU":
            cv2.rectangle(frame, (50, 50), (WIDTH-50, HEIGHT-50), (0, 0, 0), cv2.FILLED)
            draw_centered_text(frame, "SIMULADOR DE HACKEO", cv2.FONT_HERSHEY_SIMPLEX, 1.0, GREEN, 3, 100)
            draw_centered_text(frame, "Escribe tu nombre con el teclado:", cv2.FONT_HERSHEY_SIMPLEX, 0.6, WHITE, 2, 160)
            
            name_display = self.player_name + "_" if time.time() % 1 > 0.5 else self.player_name
            draw_centered_text(frame, name_display, cv2.FONT_HERSHEY_SIMPLEX, 1.2, RED_TARGET, 2, 220)
            
            bx, by, bw, bh = self.btn_start
            cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), GREEN, 2)
            cv2.putText(frame, "INICIAR", (bx + 35, by + 27), cv2.FONT_HERSHEY_SIMPLEX, 0.7, GREEN, 2)
            
            for px, py in pinch_points:
                if check_button_click(px, py, bx, by, bw, bh):
                    self.start_game()
            return

        # --- ESTADO: FIN DEL JUEGO (HISTORIAL) ---
        if self.state == "END":
            cv2.rectangle(frame, (40, 40), (WIDTH-40, HEIGHT-40), (0, 0, 0), cv2.FILLED)
            draw_centered_text(frame, "HACKEO COMPLETADO", cv2.FONT_HERSHEY_SIMPLEX, 1.0, GREEN, 3, 80)
            
            # Formato requerido: Nombre y Tiempo Total
            draw_centered_text(frame, f"Hacker: {self.player_name}", cv2.FONT_HERSHEY_SIMPLEX, 0.8, WHITE, 2, 130)
            draw_centered_text(frame, f"Tiempo Total: {sum(self.level_times):.2f} seg", cv2.FONT_HERSHEY_SIMPLEX, 0.8, YELLOW, 2, 170)
            
            # Ranking de los mejores
            draw_centered_text(frame, "--- RANKING ---", cv2.FONT_HERSHEY_SIMPLEX, 0.6, RED_TARGET, 2, 230)
            for idx, (hist_name, hist_time) in enumerate(self.history[:3]):
                texto = f"{idx+1}. {hist_name} - {hist_time:.2f}s"
                draw_centered_text(frame, texto, cv2.FONT_HERSHEY_SIMPLEX, 0.7, WHITE, 2, 270 + (idx*35))
            
            # Botón NUEVO JUEGO
            bx, by, bw, bh = self.btn_restart
            cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), GREEN, 2)
            cv2.putText(frame, "NUEVO JUEGO", (bx + 25, by + 27), cv2.FONT_HERSHEY_SIMPLEX, 0.7, GREEN, 2)
            
            for px, py in pinch_points:
                if check_button_click(px, py, bx, by, bw, bh):
                    self.reset_to_menu()
            return

        # --- ESTADO: TRANSICIÓN ENTRE NIVELES ---
        if self.state == "TRANSITION":
            self.transition_timer -= 1
            tiempo_nivel = self.level_times[-1]
            draw_centered_text(frame, f"NIVEL {self.current_level} COMPLETADO", cv2.FONT_HERSHEY_SIMPLEX, 1.0, GREEN, 3, HEIGHT // 2 - 20)
            draw_centered_text(frame, f"Tiempo del Nivel: {tiempo_nivel:.2f}s", cv2.FONT_HERSHEY_SIMPLEX, 0.8, YELLOW, 2, HEIGHT // 2 + 20)
            
            if self.transition_timer <= 0:
                self.setup_level() # <-- CORRECCIÓN: Delegate the state change logic to setup_level
            return

        # --- ESTADO: JUGANDO ---
        if self.state == "PLAYING":
            tiempo_actual = time.time() - self.level_start_time
            
            if is_paused:
                draw_centered_text(frame, "SISTEMA EN PAUSA", cv2.FONT_HERSHEY_SIMPLEX, 1.2, YELLOW, 3, HEIGHT // 2)
                tx, ty = int(self.target.x), int(self.target.y)
                if 0 < ty < HEIGHT:
                    cv2.putText(frame, self.target.char, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 1.0, RED_TARGET, 3)
            else:
                self.target.update()
                tx, ty = int(self.target.x), int(self.target.y)
                if 0 < ty < HEIGHT:
                    cv2.putText(frame, self.target.char, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 1.0, RED_TARGET, 3)
                    
                    for px, py in pinch_points:
                        if math.hypot(tx - px, ty - py) < 35: 
                            self.current_char_index += 1
                            if self.current_char_index < len(self.current_word) and self.current_word[self.current_char_index] == " ":
                                self.current_char_index += 1 
                                
                            if self.current_char_index < len(self.current_word):
                                self.target = TargetLetter(self.current_word[self.current_char_index], self.current_level)
                            else:
                                # Nivel superado, guardar tiempo individual
                                self.level_times.append(tiempo_actual)
                                self.current_level += 1
                                self.state = "TRANSITION"
                                self.transition_timer = 90
                            break 

            # HUD Superior
            progress_str = ""
            for i in range(len(self.current_word)):
                if self.current_word[i] == " ": progress_str += "  " 
                elif i < self.current_char_index: progress_str += self.current_word[i] + " "
                else: progress_str += "_ "
            
            cv2.rectangle(frame, (0, 0), (WIDTH, 80), (0, 0, 0), cv2.FILLED)
            cabecera = f"NIVEL: {self.current_level + 1}/3   |   TIEMPO: {tiempo_actual:.2f}s"
            draw_centered_text(frame, cabecera, cv2.FONT_HERSHEY_SIMPLEX, 0.6, WHITE, 2, 30)
            draw_centered_text(frame, f"OBJETIVO: {progress_str}", cv2.FONT_HERSHEY_SIMPLEX, 0.8, RED_TARGET, 2, 65)

# --- INICIALIZACIÓN ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)

game_system = GameSystem()

# --- CENTRAR LA VENTANA EN TU MONITOR ---
root = tk.Tk()
x_pos = (root.winfo_screenwidth() - WIDTH) // 2
y_pos = (root.winfo_screenheight() - HEIGHT) // 2
root.destroy() 

nombre_ventana = "Hackeando el Sistema"
cv2.namedWindow(nombre_ventana)
cv2.moveWindow(nombre_ventana, x_pos, y_pos)

exit_btn = (WIDTH - 90, 10, 75, 30)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        continue

    # --- LECTURA DE TECLADO ---
    key = cv2.waitKey(1)
    if key != -1:
        if key == 27: # ESC
            break
        elif game_system.state == "MENU":
            if key == 8 or key == 127: # Backspace
                game_system.player_name = game_system.player_name[:-1]
            elif key == 13: # Enter
                game_system.start_game()
            elif 32 <= key <= 126 and len(game_system.player_name) < 12: 
                game_system.player_name += chr(key).upper()

    frame = cv2.resize(frame, (WIDTH, HEIGHT))
    frame = cv2.flip(frame, 1)
    matrix_frame = np.zeros_like(frame)

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)

    current_frame_pinch_points = []
    juego_pausado = False
    cerrar_juego = False

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            if is_hand_open(hand_landmarks, WIDTH, HEIGHT):
                juego_pausado = True

            pinch = get_pinch_point(hand_landmarks, WIDTH, HEIGHT)
            if pinch:
                px, py = pinch
                cv2.circle(frame, (px, py), 10, YELLOW, cv2.FILLED)
                
                if check_button_click(px, py, *exit_btn):
                    cerrar_juego = True
                else:
                    current_frame_pinch_points.append((px, py))

    if cerrar_juego:
        break

    game_system.draw_rain(matrix_frame, juego_pausado)
    combined_frame = cv2.addWeighted(frame, 0.4, matrix_frame, 0.6, 0)
    
    cv2.rectangle(combined_frame, (exit_btn[0], exit_btn[1]), (exit_btn[0]+exit_btn[2], exit_btn[1]+exit_btn[3]), RED_TARGET, cv2.FILLED)
    cv2.putText(combined_frame, "SALIR", (exit_btn[0] + 12, exit_btn[1] + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 2)
    
    game_system.update_and_draw_game(matrix_frame, combined_frame, current_frame_pinch_points, juego_pausado)

    cv2.imshow(nombre_ventana, combined_frame)

cap.release()
cv2.destroyAllWindows()